<a href="https://colab.research.google.com/github/milleau98/2026-gig-data-analysis/blob/main/notebooks/dataset-generators/google_trends.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pytrends pandas


In [2]:
from pytrends.request import TrendReq
import pandas as pd
import time

# connect to Google trends
pytrends = TrendReq(hl='en-US', tz=360)

anchor = 'Uber'

keyword_groups = [
    [anchor,'Lyft','DoorDash','Instacart'],
    [anchor, 'Fiverr','Upwork','side hustle','gig'],
    [anchor, 'Herbalife','Primerica','Nu skin', 'USANA'],
    [anchor, 'Etsy', 'Shopify', 'Udemy', 'Tupperware']
]

all_data = []

for i, group in enumerate(keyword_groups):
  # timeframe and geo
  pytrends.build_payload(group, timeframe='2010-01-01 2025-12-31', geo='US')
  time.sleep(5)

  df=pytrends.interest_over_time()

  df=df.drop(columns=['isPartial'])

  # store anchor values
  if i == 0:
    anchor_series = df[anchor]
    all_data.append(df)
  else:
    #scale_factor = anchor_series.div(df[anchor]).replace([float("inf"), -float("inf")], 0)
    scale_factor = anchor_series.div(df[anchor]).replace([float("inf"), -float("inf")], float("nan"))
    df = df.multiply(scale_factor, axis=0)
    all_data.append(df.drop(columns=[anchor]))

# combine groups of data
final_data = pd.concat(all_data, axis=1)

final_data.head()

,Uber,Lyft,DoorDash,Instacart,Fiverr,Upwork,side hustle,gig,Herbalife,Primerica,Nu skin,USANA,Etsy,Shopify,Udemy,Tupperware
date,,,,,,,,,,,,,,,,
2010-01-01,2,0,0,0,0.0,0.0,0.0,7.0,3.0,2.0,1.0,1.0,13.0,0.0,0.0,3.0
2010-02-01,2,0,0,0,0.0,0.0,0.0,7.0,3.0,2.0,1.0,1.0,13.0,0.0,0.0,2.0
2010-03-01,2,0,0,0,0.0,0.0,0.0,7.0,3.0,3.0,1.0,1.0,13.0,0.0,0.0,3.0
2010-04-01,2,0,0,0,0.0,0.0,0.0,7.0,3.0,3.0,0.0,1.0,13.0,0.0,0.0,2.0
2010-05-01,2,0,0,0,0.0,0.0,0.0,7.0,3.0,2.0,0.0,1.0,13.0,0.0,0.0,3.0


In [3]:
# convert weekly trends to monthly averages
monthly = final_data.resample("M").mean()

monthly['year'] = monthly.index.year
monthly['month'] = monthly.index.month
monthly['date'] = monthly.index

monthly = monthly.reset_index(drop=True)

monthly.head()

/tmp/ipykernel_7035/1836030185.py:2: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly = final_data.resample("M").mean()


,Uber,Lyft,DoorDash,Instacart,Fiverr,Upwork,side hustle,gig,Herbalife,Primerica,Nu skin,USANA,Etsy,Shopify,Udemy,Tupperware,year,month,date
0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,3.0,2.0,1.0,1.0,13.0,0.0,0.0,3.0,2010,1,2010-01-31
1,2.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,3.0,2.0,1.0,1.0,13.0,0.0,0.0,2.0,2010,2,2010-02-28
2,2.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,3.0,3.0,1.0,1.0,13.0,0.0,0.0,3.0,2010,3,2010-03-31
3,2.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,3.0,3.0,0.0,1.0,13.0,0.0,0.0,2.0,2010,4,2010-04-30
4,2.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,3.0,2.0,0.0,1.0,13.0,0.0,0.0,3.0,2010,5,2010-05-31


In [5]:
import os
os.makedirs("data/google_trends", exist_ok=True)

# create dataset files
monthly.to_csv('data/google_trends/google_trends_monthly.csv', index=False)

In [6]:
# download as csv
from google.colab import files
files.download('data/google_trends/google_trends_monthly.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>